# Assignment 2 - Game 1: Quoridor

## Instructions

This is a **self-contained notebook** - everything you need is here!

### Quick Start
1. **Run all cells** up to Section 4 (this loads the game client)
2. **Implement your solver** in Section 5
3. **Update configuration** in Section 7
4. **Play the game** in Section 7

### What You Need To Do
- Focus ONLY on implementing `my_agent()` function (Section 5)
  - You can create various players, you'll be able to select your preferred one.
- Everything else is provided for you!

### About Quoridor
Quoridor is a 2-player abstract strategy game where:
- Players start at opposite ends of a 9x9 board
- Goal: Be the first to reach the opposite side
- Each turn: Either move your pawn OR place a wall to block opponent
- Each player has 10 walls to use during the game
- Walls must not completely block any player from reaching their goal

---
## Section 1: Setup

**Run this cell (no changes needed)**

In [1]:
import requests
import json
import time
import random
from typing import List, Optional, Tuple, Any, Dict
from copy import deepcopy
from collections import deque

print("✅ Dependencies imported")

BASE_URL = 'https://ie-aireasoning-gr4r5bl6tq-ew.a.run.app'  # Your Cloud Run URL

print("✅ Configuration loaded")

✅ Dependencies imported
✅ Configuration loaded


---
## Section 2: Game Client Library

**Run this cell (no changes needed)**

This defines the game client that handles all server communication.

In [3]:
class GameClient:
    def __init__(self, base_url: str, token: str, debug: bool = False):
        self.base_url = base_url.rstrip('/')
        self.token = token
        self.debug = debug

    def _make_request(self, endpoint: str, params: dict, max_retries: int = 10) -> dict:
        params['TOKEN'] = self.token
        url = f'{self.base_url}{endpoint}'

        for attempt in range(max_retries):
            try:
                if self.debug:
                    print(f"[DEBUG] Request: {endpoint}")
                    print(f"[DEBUG] Params: {params}")

                response = requests.get(url, params=params, timeout=30)

                if self.debug:
                    print(f"[DEBUG] Response [{response.status_code}]: {response.text[:200]}")

                if response.status_code == 200:
                    if response.text:
                        try:
                            return response.json()
                        except (json.JSONDecodeError, ValueError) as e:
                            if self.debug:
                                print(f"[DEBUG] Non-JSON response: {response.text[:100]}")
                            return {}
                    return {}
                else:
                    print(f"⚠️  HTTP {response.status_code}: {response.text[:200]}")

            except requests.exceptions.Timeout:
                print(f"⚠️  Request timeout (attempt {attempt + 1}/{max_retries})")
            except requests.exceptions.RequestException as e:
                print(f"⚠️  Request error: {e} (attempt {attempt + 1}/{max_retries})")
            except Exception as e:
                print(f"⚠️  Unexpected error: {type(e).__name__}: {e} (attempt {attempt + 1}/{max_retries})")

            if attempt < max_retries - 1:
                time.sleep(1)

        raise Exception(f"Failed to connect to {endpoint} after {max_retries} attempts")

    def create_match(self, game_type: str, num_games: int, multiplayer: bool = False) -> str:
        response = self._make_request('/new-match', {
            'game-type': game_type,
            'num-games': str(num_games),
            'multi-player': 'True' if multiplayer else 'False'
        })

        if 'match-id' not in response:
            print(f"❌ Server response missing 'match-id'. Response: {response}")
            raise KeyError(f"Server response missing 'match-id'. Got: {response}")

        return response['match-id']

    def join_match(self, match_id: str) -> dict:
        response = self._make_request('/join-match', {
            'match-id': match_id
        })
        return response

    def get_game_state(self, match_id: str, game_index: int) -> dict:
        return self._make_request('/game-state-in-match', {
            'match-id': match_id,
            'game-index': str(game_index)
        })

    def get_match_state(self, match_id: str) -> dict:
        return self._make_request('/match-state', {
            'match-id': match_id
        })

    def make_move(self, match_id: str, player: str, move: Any) -> bool:
        move_str = move if isinstance(move, str) else json.dumps(move)

        self._make_request('/make-move-in-match', {
            'match-id': match_id,
            'player': player,
            'move': move_str
        })
        return True

print("✅ GameClient loaded")


def play_game(
    solver,
    base_url: str,
    token: str,
    game_type: str,
    game_class,
    multiplayer: bool = False,
    match_id: Optional[str] = None,
    num_games: int = 1,
    debug: bool = False,
    verbose: bool = True
) -> Tuple:
    client = GameClient(base_url, token, debug=debug)

    if match_id is None:
        if verbose:
            print(f"🎮 Creating new match: {num_games} x {game_type}")
        match_id = client.create_match(game_type, num_games, multiplayer)
        if verbose:
            print(f"   Match ID: {match_id}")

    if verbose:
        print(f"🔗 Joining match {match_id}...")
    match = client.join_match(match_id)
    player = match['player']
    num_games = match.get('num-games', num_games)
    if verbose:
        print(f"   You are player: {player}")

    game_state = client.get_game_state(match_id, 0)
    if game_state['status'] == 'waiting':
        if verbose:
            print("⏳ Waiting for opponent to join...")
        while game_state['status'] == 'waiting':
            time.sleep(2)
            game_state = client.get_game_state(match_id, 0)

    all_results = []
    wins = 0
    losses = 0
    draws = 0

    while True:
        match_state = client.get_match_state(match_id)
        if match_state['status'] != 'in_progress':
            break
        game_num = match_state['current-game-index']

        if verbose:
            print(f"\n{'='*50}")
            print(f"🎮 GAME {game_num + 1}/{num_games}")
            print(f"{'='*50}\n")

        # Get initial game state and check player assignment
        game_state = client.get_game_state(match_id, game_num)

        # Update player sign if it changed (randomized per game)
        if 'my-player' in game_state and game_state['my-player']:
            new_player = game_state['my-player']
            if new_player != player and verbose and game_num > 0:
                print(f"ℹ️  Player assignment changed: You are now Player {new_player}\n")
            player = new_player

        game = game_class(game_state['state'], game_state['status'], game_state['player'], player)

        move_count = 0
        while game_state['status'] != 'complete':
            game_state = client.get_game_state(match_id, game_num)
            player = game_state['my-player']
            if 'winner' in game_state:
                break

            game = game_class(game_state['state'], game_state['status'], game_state['player'], player)

            if game.is_terminal():
                break

            if verbose:
                game.print_board()

            if game.current_player == player:
                if verbose:
                    print(f"🤔 Your turn (Player {player})...")

                try:
                    move = solver(game)

                    if verbose:
                        if hasattr(move, '__iter__') and not isinstance(move, str):
                            if len(move) > 0 and move[0] == 'M':
                                print(f"   Moving pawn to ({move[1]}, {move[2]})")
                            elif len(move) > 0 and move[0] == 'W':
                                print(f"   Placing {move[3]} wall at ({move[1]}, {move[2]})")
                            else:
                                print(f"   Move: {move}")
                        else:
                            print(f"   Move: {move}")

                    client.make_move(match_id, player, move)
                    move_count += 1

                except Exception as e:
                    print(f"❌ Error in solver: {e}")
                    import traceback
                    traceback.print_exc()
                    all_results.append(('error', None))
                    break
            else:
                if verbose:
                    print(f"⏳ Waiting for opponent (Player {game.current_player})...")
                time.sleep(2)

        # game is already terminal from the loop above
        if verbose:
            game.print_board()
            print("=" * 40)

        winner = game_state['winner']
        if winner == '-':
            if verbose:
                print("🤝 Game ended in a DRAW!")
            result = 'draw'
            draws += 1
        elif winner == player:
            if verbose:
                print("🎉 You WON! Congratulations!")
            result = 'win'
            wins += 1
        else:
            if verbose:
                print("😞 You LOST. Better luck next time!")
            result = 'loss'
            losses += 1

        all_results.append((result, player, winner))

        if verbose and num_games > 1:
            print(f"\n📊 Current Record: {wins}W - {losses}L - {draws}D")
            print(f"   Games Remaining: {num_games - game_num - 1}\n")

    # Return results
    stats = {
        'wins': wins,
        'losses': losses,
        'draws': draws,
        'total_games': num_games,
        'win_rate': wins / num_games if num_games > 0 else 0,
        'player': player,
        'match_id': match_id
    }

    return stats, all_results

print("✅ play_game loaded")

✅ GameClient loaded
✅ play_game loaded


---
## Section 3: Game State Class & Helper Functions

**Run this cell (no changes needed)**

This defines the `QuoridorGame` class with all helper methods you'll need.

In [4]:
class QuoridorGame:
    """
    Represents Quoridor game state with helper methods.

    Key methods for your solver:
    - game.get_my_position()           # Your pawn position [row, col]
    - game.get_opponent_position()     # Opponent pawn position [row, col]
    - game.get_walls()                 # List of walls [[r, c, 'H'/'V'], ...]
    - game.get_remaining_walls()       # Your remaining wall count
    - game.get_valid_moves()           # All valid moves
    - game.get_valid_pawn_moves()      # Only pawn moves
    - game.get_valid_wall_moves()      # Only wall placements
    - game.simulate_move(move)         # Simulate move for search
    - game.is_terminal()               # Check if game over
    - game.print_board()               # Debug visualization
    """

    def __init__(self, state: str, status: str, current_player: str, my_player: str):
        self.state_str = state
        self.status = status
        self.current_player = current_player
        self.my_player = my_player
        self._state = None
        self._valid_moves = None

    @property
    def state(self) -> Dict:
        """Get state dictionary."""
        if self._state is None:
            self._state = json.loads(self.state_str)
        return self._state

    def get_my_position(self) -> List[int]:
        """Get your pawn position [row, col]."""
        return self.state['pawns'][self.my_player]

    def get_opponent_position(self) -> List[int]:
        """Get opponent pawn position [row, col]."""
        opp = '2' if self.my_player == '1' else '1'
        return self.state['pawns'][opp]

    def get_walls(self) -> List[List]:
        """Get list of walls [[row, col, 'H'/'V'], ...]."""
        return self.state['walls']

    def get_remaining_walls(self, player: Optional[str] = None) -> int:
        """Get remaining wall count for player (defaults to you)."""
        if player is None:
            player = self.my_player
        return self.state['remaining_walls'][player]

    def is_terminal(self) -> bool:
        """Check if game is over."""
        return self.status == 'complete'

    def is_waiting(self) -> bool:
        """Check if waiting for opponent."""
        return self.status == 'waiting'

    def get_winner(self) -> Optional[str]:
        """Get winner ('1', '2', None if ongoing)."""
        if not self.is_terminal():
            return None
        return self.current_player

    def get_opponent(self, player: str) -> str:
        """Get opponent's identifier."""
        return '2' if player == '1' else '1'

    def _is_wall_blocking(self, walls, from_pos, to_pos) -> bool:
        """Check if a wall blocks movement."""
        r1, c1 = from_pos
        r2, c2 = to_pos

        def has_wall(r, c, orient):
            return [r, c, orient] in walls

        # Moving up
        if r2 < r1 and c1 == c2:
            return has_wall(r2, c1, 'H') or has_wall(r2, c1 - 1, 'H')
        # Moving down
        elif r2 > r1 and c1 == c2:
            return has_wall(r1, c1, 'H') or has_wall(r1, c1 - 1, 'H')
        # Moving left
        elif c2 < c1 and r1 == r2:
            return has_wall(r1, c2, 'V') or has_wall(r1 - 1, c2, 'V')
        # Moving right
        elif c2 > c1 and r1 == r2:
            return has_wall(r1, c1, 'V') or has_wall(r1 - 1, c1, 'V')

        return False

    def get_valid_pawn_moves(self, player: Optional[str] = None) -> List[List]:
        """Get valid pawn moves for player."""
        if player is None:
            player = self.current_player

        state = self.state
        my_pos = state['pawns'][player]
        opp_pos = state['pawns'][self.get_opponent(player)]
        walls = state['walls']

        r, c = my_pos
        moves = []

        # Basic adjacent moves
        adjacent = [(r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)]

        for new_r, new_c in adjacent:
            if 0 <= new_r < 9 and 0 <= new_c < 9:
                if [new_r, new_c] == opp_pos:
                    # Try to jump over opponent
                    dr = new_r - r
                    dc = new_c - c
                    jump_r = new_r + dr
                    jump_c = new_c + dc

                    if 0 <= jump_r < 9 and 0 <= jump_c < 9:
                        if not self._is_wall_blocking(walls, opp_pos, [jump_r, jump_c]):
                            moves.append(['M', jump_r, jump_c])
                    else:
                        # Diagonal jumps when can't jump straight
                        if dr != 0:  # Moving vertically, try left/right
                            for side_dc in [-1, 1]:
                                side_c = new_c + side_dc
                                if 0 <= side_c < 9:
                                    if not self._is_wall_blocking(walls, opp_pos, [new_r, side_c]):
                                        moves.append(['M', new_r, side_c])
                        else:  # Moving horizontally, try up/down
                            for side_dr in [-1, 1]:
                                side_r = new_r + side_dr
                                if 0 <= side_r < 9:
                                    if not self._is_wall_blocking(walls, opp_pos, [side_r, new_c]):
                                        moves.append(['M', side_r, new_c])
                else:
                    # Normal move
                    if not self._is_wall_blocking(walls, my_pos, [new_r, new_c]):
                        moves.append(['M', new_r, new_c])

        return moves

    def get_valid_wall_moves(self, player: Optional[str] = None, limit: int = None) -> List[List]:
        """
        Get valid wall placements for player.

        Args:
            player: Player to get walls for (defaults to current player)
            limit: Maximum number of wall moves to return (for efficiency)
        """
        if player is None:
            player = self.current_player

        if self.get_remaining_walls(player) == 0:
            return []

        # NOTE: Checking all wall placements is expensive!
        # For performance, you may want to limit the number checked
        # or implement strategic wall placement heuristics

        moves = []
        walls = self.get_walls()

        # This is computationally expensive - consider limiting in your implementation
        # Try horizontal walls (simplified - only checks some positions)
        if limit:
            # Sample some random positions instead of checking all
            positions = [(r, c) for r in range(9) for c in range(8)]
            random.shuffle(positions)
            positions = positions[:limit]

            for r, c in positions:
                if self._is_valid_wall_simple([r, c, 'H'], walls):
                    moves.append(['W', r, c, 'H'])

            positions = [(r, c) for r in range(8) for c in range(9)]
            random.shuffle(positions)
            positions = positions[:limit]

            for r, c in positions:
                if self._is_valid_wall_simple([r, c, 'V'], walls):
                    moves.append(['W', r, c, 'V'])
        else:
            # Check all positions (slow!)
            for r in range(9):
                for c in range(8):
                    if self._is_valid_wall_simple([r, c, 'H'], walls):
                        moves.append(['W', r, c, 'H'])

            for r in range(8):
                for c in range(9):
                    if self._is_valid_wall_simple([r, c, 'V'], walls):
                        moves.append(['W', r, c, 'V'])

        return moves

    def _is_valid_wall_simple(self, wall, existing_walls) -> bool:
        """Simplified wall validation (doesn't check pathfinding)."""
        r, c, orient = wall

        # Check if wall already exists
        if wall in existing_walls:
            return False

        # Check for overlaps and crossings (simplified)
        if orient == 'H':
            if [r, c + 1, 'H'] in existing_walls or [r, c - 1, 'H'] in existing_walls:
                return False
            if [r - 1, c, 'V'] in existing_walls or [r, c, 'V'] in existing_walls:
                return False
            if [r - 1, c + 1, 'V'] in existing_walls or [r, c + 1, 'V'] in existing_walls:
                return False
        else:  # 'V'
            if [r + 1, c, 'V'] in existing_walls or [r - 1, c, 'V'] in existing_walls:
                return False
            if [r, c - 1, 'H'] in existing_walls or [r, c, 'H'] in existing_walls:
                return False
            if [r + 1, c - 1, 'H'] in existing_walls or [r + 1, c, 'H'] in existing_walls:
                return False

        # NOTE: This doesn't check if wall blocks all paths!
        # The server will reject invalid walls, but checking here is expensive
        return True

    def get_valid_moves(self, player: Optional[str] = None, limit_walls: int = 20) -> List[List]:
        """
        Get all valid moves.

        Args:
            player: Player to get moves for
            limit_walls: Limit wall checks for performance (set to None for all)
        """
        if self._valid_moves is None or player != self.current_player:
            pawn_moves = self.get_valid_pawn_moves(player)
            wall_moves = self.get_valid_wall_moves(player, limit=limit_walls)
            self._valid_moves = pawn_moves + wall_moves
        return self._valid_moves

    def simulate_move(self, move: List) -> Dict:
        """
        Simulate a move and return new state.
        Does NOT contact server or modify original state.
        Essential for minimax/alpha-beta, but this can be expensive because of deepcopy.
        """
        new_state = deepcopy(self.state)
        player = self.current_player

        if move[0] == 'M':
            # Pawn move
            new_state['pawns'][player] = [move[1], move[2]]
        elif move[0] == 'W':
            # Wall placement
            new_state['walls'].append([move[1], move[2], move[3]])
            new_state['remaining_walls'][player] -= 1

        return new_state

    def shortest_path_length(self, player: Optional[str] = None, state: Optional[Dict] = None) -> int:
        """
        Calculate shortest path length to goal using BFS.
        Useful for evaluation functions!

        Returns:
            int: Number of moves to goal, or 999 if no path
        """
        if player is None:
            player = self.my_player
        if state is None:
            state = self.state

        start = tuple(state['pawns'][player])
        goal_row = 8 if player == '1' else 0
        walls = state['walls']

        visited = {start: 0}
        queue = deque([start])

        while queue:
            r, c = queue.popleft()
            dist = visited[(r, c)]

            if r == goal_row:
                return dist

            for new_r, new_c in [(r-1, c), (r+1, c), (r, c-1), (r, c+1)]:
                if 0 <= new_r < 9 and 0 <= new_c < 9:
                    if (new_r, new_c) not in visited:
                        if not self._is_wall_blocking(walls, [r, c], [new_r, new_c]):
                            visited[(new_r, new_c)] = dist + 1
                            queue.append((new_r, new_c))

        return 999  # No path found

    def print_board(self):
        """Print nice board visualization."""
        state = self.state
        pawns = state['pawns']
        walls = state['walls']

        print("\n" + "=" * 37)
        print(f"Player 1: {pawns['1']}  Player 2: {pawns['2']}")
        print(f"Walls remaining - You: {self.get_remaining_walls(self.my_player)}, " +
              f"Opponent: {self.get_remaining_walls(self.get_opponent(self.my_player))}")
        print("=" * 37)

        # Create visual board
        for r in range(9):
            # Print row with cells
            row_str = ""
            for c in range(9):
                if pawns['1'] == [r, c]:
                    row_str += "1"
                elif pawns['2'] == [r, c]:
                    row_str += "2"
                else:
                    row_str += "·"

                # Check for vertical wall to the right
                if c < 8:
                    if [r, c, 'V'] in walls or (r > 0 and [r - 1, c, 'V'] in walls):
                        row_str += " ║ "
                    else:
                        row_str += "   "

            print(row_str)

            # Print horizontal walls
            if r < 8:
                wall_str = ""
                for c in range(9):
                    if [r, c, 'H'] in walls or (c > 0 and [r, c - 1, 'H'] in walls):
                        wall_str += "═"
                    else:
                        wall_str += " "

                    if c < 8:
                        wall_str += "   "

                print(wall_str)

        print("=" * 37 + "\n")

---
## Section 4: Manual Play Mode (Try the Game Yourself!)

**Play Quoridor manually** to understand the game before implementing your AI!

This lets you:
- Experience the game firsthand
- Test the server connection
- Understand winning strategies
- Play against the server AI

In [5]:
def manual_player_solver(game: QuoridorGame) -> List:
    """
    Interactive manual player - YOU choose the moves!
    Perfect for testing the game and understanding the rules.
    """
    game.print_board()

    pawn_moves = game.get_valid_pawn_moves()
    wall_moves = game.get_valid_wall_moves(limit=30)

    print(f"\n🎮 YOUR TURN (Player {game.my_player})!")
    print("\nValid pawn moves:")
    for i, move in enumerate(pawn_moves):
        print(f"  {i}: Move to ({move[1]}, {move[2]})")

    if wall_moves and game.get_remaining_walls() > 0:
        print(f"\nOr place a wall (you have {game.get_remaining_walls()} left):")
        print("  Enter: W row col H/V (e.g., 'W 3 4 H')")

    while True:
        try:
            choice = input("\nEnter your choice (number for pawn move, or 'W r c H/V' for wall): ").strip()

            if choice.lower() == 'q':
                raise KeyboardInterrupt()

            if choice.upper().startswith('W'):
                parts = choice.split()
                if len(parts) == 4:
                    move = ['W', int(parts[1]), int(parts[2]), parts[3].upper()]
                    return move
                else:
                    print("❌ Invalid format! Use: W row col H/V")
            else:
                idx = int(choice)
                if 0 <= idx < len(pawn_moves):
                    return pawn_moves[idx]
                else:
                    print(f"❌ Invalid index! Choose 0-{len(pawn_moves)-1}")

        except ValueError:
            print("❌ Invalid input! Enter a number or 'W row col H/V'")
        except KeyboardInterrupt:
            print("\n👋 Thanks for playing!")
            raise

print("✅ Manual player loaded")
print("   Run the cell below to play interactively!")

✅ Manual player loaded
   Run the cell below to play interactively!


---
## Section 5: YOUR SOLVER IMPLEMENTATION

**⭐ THIS IS WHERE YOU WRITE YOUR CODE! ⭐**

Implement your AI algorithm here. You can use:
- Minimax
- Alpha-beta pruning
- Custom heuristics
- A* pathfinding

### Available Methods

```python
game.get_my_position()                    # Your pawn [row, col]
game.get_opponent_position()              # Opponent pawn [row, col]
game.get_walls()                          # List of walls
game.get_remaining_walls(player)          # Wall count
game.get_valid_moves()                    # All valid moves
game.get_valid_pawn_moves()               # Only pawn moves
game.get_valid_wall_moves(limit=20)       # Only walls (limited for speed)
game.simulate_move(move)                  # Simulate move, returns new state
game.shortest_path_length(player, state)  # BFS path length to goal
game.print_board()                        # Print board
```

### Move Format
- Pawn move: `['M', row, col]`
- Wall placement: `['W', row, col, 'H']` or `['W', row, col, 'V']`

### Strategy Tips
1. Use `shortest_path_length()` in your evaluation function
2. Balance pawn advancement vs. blocking opponent with walls
3. Be careful with wall placement - checking all positions is slow!
4. Consider limiting search depth due to high branching factor

In [5]:
'''

def my_agent(game: QuoridorGame) -> List:
    """
    Your AI implementation.

    Args:
        game: QuoridorGame object with helper methods

    Returns:
        List: Move in format ['M', row, col] or ['W', row, col, 'H'/'V']
    """

    # Get basic info
    my_player = game.my_player
    my_pos = game.get_my_position()
    opp_pos = game.get_opponent_position()

    # ============================================================
    # TODO: IMPLEMENT YOUR ALGORITHM HERE!
    # ============================================================

    # Example: Random valid move (replace with your algorithm!)
    moves = game.get_valid_pawn_moves() + game.get_valid_wall_moves(limit=10)

    # Just pick a random pawn move
    return random.choice(moves)

    # ============================================================
    # Ideas to implement:
    # 1. Use shortest_path_length() to evaluate positions
    # 2. Alpha-beta minimax with evaluation heuristic (e.g. my_path_length - opponent_path_length)
    # 4. Strategic wall placement to maximize opponent's path
    # ============================================================

print("✅ Solver function defined")
print("   Remember to implement your algorithm before running!")

'''

'\n\ndef my_agent(game: QuoridorGame) -> List:\n    """\n    Your AI implementation.\n    \n    Args:\n        game: QuoridorGame object with helper methods\n    \n    Returns:\n        List: Move in format [\'M\', row, col] or [\'W\', row, col, \'H\'/\'V\']\n    """\n    \n    # Get basic info\n    my_player = game.my_player\n    my_pos = game.get_my_position()\n    opp_pos = game.get_opponent_position()\n    \n    # ============================================================\n    # TODO: IMPLEMENT YOUR ALGORITHM HERE!\n    # ============================================================\n    \n    # Example: Random valid move (replace with your algorithm!)\n    moves = game.get_valid_pawn_moves() + game.get_valid_wall_moves(limit=10)\n        \n    # Just pick a random pawn move\n    return random.choice(moves)\n    \n    # ============================================================\n    # Ideas to implement:\n    # 1. Use shortest_path_length() to evaluate positions\n    # 2. Alpha-

In [ ]:
def my_agent(game: QuoridorGame) -> List:
    import random
    import json
    from copy import deepcopy

    me = game.my_player
    opp = game.get_opponent(me)
    root_state = deepcopy(game.state)

    # -------------------------------------------
    # Helper: fully verify a wall is server-legal
    # -------------------------------------------
    def is_fully_legal_wall(move):
        if move[0] != 'W':
            return True

        r, c, o = move[1], move[2], move[3]

        # Simulate applying wall
        new_state = deepcopy(root_state)
        new_state["walls"] = new_state["walls"] + [[r, c, o]]

        # Test path existence for both players
        d_me = game.shortest_path_length(player=me, state=new_state)
        d_opp = game.shortest_path_length(player=opp, state=new_state)

        # If either player loses all paths → illegal
        if d_me >= 999 or d_opp >= 999:
            return False

        return True

    # -------------------------------------------
    # If terminal, just return any legal pawn move
    # -------------------------------------------
    if game.is_terminal():
        pawn_moves = game.get_valid_pawn_moves()
        if pawn_moves:
            return random.choice(pawn_moves)
        all_moves = game.get_valid_moves()
        if all_moves:
            return random.choice(all_moves)
        pos = game.get_my_position()
        return ['M', pos[0], pos[1]]

    # -------------------------------------------
    # State simulation
    # -------------------------------------------
    def apply_move(state, player, move):
        s = deepcopy(state)
        if move[0] == 'M':
            s['pawns'][player] = [move[1], move[2]]
        else:
            s['walls'] = s['walls'] + [[move[1], move[2], move[3]]]
            s['remaining_walls'][player] -= 1
        return s

    # -------------------------------------------
    # Evaluation function
    # -------------------------------------------
    def eval_state(state):
        my_pos = state['pawns'][me]
        opp_pos = state['pawns'][opp]

        my_d = game.shortest_path_length(player=me, state=state)
        opp_d = game.shortest_path_length(player=opp, state=state)

        if my_d == 0: return 10_000
        if opp_d == 0: return -10_000
        if my_d >= 999 and opp_d >= 999: return 0
        if my_d >= 999: return -9_000
        if opp_d >= 999: return 9_000

        my_rw = state['remaining_walls'][me]
        opp_rw = state['remaining_walls'][opp]

        center_col = 4
        my_center = -abs(my_pos[1] - center_col)
        opp_center = -abs(opp_pos[1] - center_col)

        race_term = 3.0 * (opp_d - my_d)
        walls_term = 0.5 * (my_rw - opp_rw)
        center_term = 0.2 * (my_center - opp_center)

        my_goal_row = 8 if me == '1' else 0
        opp_goal_row = 8 if opp == '1' else 0

        my_progress = -(abs(my_pos[0] - my_goal_row))
        opp_progress = abs(opp_pos[0] - opp_goal_row)
        progress_term = 0.1 * (my_progress - opp_progress)

        endgame_bonus = 0.0
        if my_d <= 2:
            endgame_bonus += 8.0
        if opp_d <= 2:
            endgame_bonus -= 8.0

        jitter = random.uniform(-0.05, 0.05)

        return race_term + walls_term + center_term + progress_term + endgame_bonus + jitter

    # -------------------------------------------
    # Candidate move generation
    # -------------------------------------------
    def get_candidate_root_moves():
        pawn_moves = game.get_valid_pawn_moves()
        wall_moves_raw = []

        if root_state['remaining_walls'][me] > 0:
            wall_moves_raw = game.get_valid_wall_moves(limit=30) or []

        my_d0 = game.shortest_path_length(player=me, state=root_state)
        opp_d0 = game.shortest_path_length(player=opp, state=root_state)

        good_walls = []
        for w in wall_moves_raw:
            s = apply_move(root_state, me, w)
            my_d = game.shortest_path_length(player=me, state=s)
            opp_d = game.shortest_path_length(player=opp, state=s)

            # still skip illegal-like situations here; final safety remains
            if opp_d >= 999 or my_d >= 999:
                continue

            gain_opp = opp_d - opp_d0
            loss_me = my_d - my_d0
            net_gain = gain_opp - loss_me

            # if I am behind in the race, allow "lighter" walls
            i_am_behind = my_d0 > opp_d0

            # core rule:
            # - net_gain >= 1 (they suffer more than me)
            # - and at least +1 on their path
            if net_gain >= 1 and gain_opp >= 1:
                good_walls.append((net_gain, gain_opp, w))
            # if I'm clearly behind, also accept small positive/net-neutral walls
            elif i_am_behind and gain_opp >= 1 and loss_me <= gain_opp:
                good_walls.append((net_gain, gain_opp, w))

        # sort walls by (net_gain, gain_opp) so best are first
        good_walls.sort(key=lambda x: (x[0], x[1]), reverse=True)
        good_walls = [w for _, _, w in good_walls[:8]]  # cap to 8 best walls

        moves = pawn_moves + good_walls

        if not moves:
            moves = game.get_valid_moves()

        return moves


    # -------------------------------------------
    # Opponent reply generation
    # -------------------------------------------
    def generate_opponent_moves(state):
        old_state = game._state
        old_cp = game.current_player
        old_vm = game._valid_moves

        game._state = deepcopy(state)
        game.current_player = opp
        game._valid_moves = None

        try:
            pawn_moves = game.get_valid_pawn_moves()
            wall_moves = []
            if state['remaining_walls'][opp] > 0:
                wall_moves = game.get_valid_wall_moves(limit=10) or []
            moves = pawn_moves + wall_moves
        except:
            moves = []
        finally:
            game._state = old_state
            game.current_player = old_cp
            game._valid_moves = old_vm

        if not moves:
            try:
                moves = game.get_valid_moves()
            except:
                moves = []

        return moves

    # -------------------------------------------
    # Minimax with depth 2
    # -------------------------------------------
    def minimax_root(moves, depth=2):
        best_score = -float('inf')
        best_moves = []

        my_d0 = game.shortest_path_length(player=me, state=root_state)
        opp_d0 = game.shortest_path_length(player=opp, state=root_state)
        near_endgame = min(my_d0, opp_d0) <= 3

        if near_endgame or depth <= 1 or len(moves) == 1:
            scored = []
            for m in moves:
                s = apply_move(root_state, me, m)
                scored.append((eval_state(s), m))
            scored.sort(key=lambda x: x[0], reverse=True)
            return scored[0][1]

        for m in moves:
            s = apply_move(root_state, me, m)
            opp_moves = generate_opponent_moves(s)

            if not opp_moves:
                score = eval_state(s)
            else:
                worst_for_me = float('inf')
                random.shuffle(opp_moves)
                for om in opp_moves[:12]:
                    s2 = apply_move(s, opp, om)
                    val = eval_state(s2)
                    if val < worst_for_me:
                        worst_for_me = val
                score = worst_for_me

            if score > best_score:
                best_score = score
                best_moves = [m]
            elif score == best_score:
                best_moves.append(m)

        return random.choice(best_moves)

    # -------------------------------------------
    # Choose move normally
    # -------------------------------------------
    root_moves = get_candidate_root_moves()
    if not root_moves:
        fallback = game.get_valid_moves()
        if fallback:
            return random.choice(fallback)
        pos = game.get_my_position()
        return ['M', pos[0], pos[1]]

    chosen = minimax_root(root_moves, depth=2)

    # -------------------------------------------
    # FINAL SAFETY CHECK:
    # Reject illegal walls before sending to server
    # -------------------------------------------
    if chosen[0] == 'W' and not is_fully_legal_wall(chosen):
        pawn_moves = game.get_valid_pawn_moves()
        if pawn_moves:
            return random.choice(pawn_moves)
        return random.choice(game.get_valid_moves())

    return chosen


---
## Section 6: Test Your Solver (Optional)

Test parts of your implementation before playing a full game.

In [7]:
# Create a test state
test_state = {
    'pawns': {'1': [2, 4], '2': [6, 4]},
    'walls': [[3, 3, 'H'], [4, 5, 'V']],
    'remaining_walls': {'1': 8, '2': 9}
}

test_game = QuoridorGame(json.dumps(test_state), 'playing', '1', '1')

print("Test board:")
test_game.print_board()

print(f"Valid pawn moves: {test_game.get_valid_pawn_moves()}")
print(f"My path length: {test_game.shortest_path_length('1')}")
print(f"Opponent path length: {test_game.shortest_path_length('2')}")

# Test your solver
move = my_agent(test_game)
print(f"\nYour solver chose: {move}")

Test board:

Player 1: [2, 4]  Player 2: [6, 4]
Walls remaining - You: 8, Opponent: 9
·   ·   ·   ·   ·   ·   ·   ·   ·
                                 
·   ·   ·   ·   ·   ·   ·   ·   ·
                                 
·   ·   ·   ·   1   ·   ·   ·   ·
                                 
·   ·   ·   ·   ·   ·   ·   ·   ·
            ═   ═                
·   ·   ·   ·   ·   · ║ ·   ·   ·
                                 
·   ·   ·   ·   ·   · ║ ·   ·   ·
                                 
·   ·   ·   ·   2   ·   ·   ·   ·
                                 
·   ·   ·   ·   ·   ·   ·   ·   ·
                                 
·   ·   ·   ·   ·   ·   ·   ·   ·

Valid pawn moves: [['M', 1, 4], ['M', 3, 4], ['M', 2, 3], ['M', 2, 5]]
My path length: 7
Opponent path length: 7

Your solver chose: ['M', 3, 4]


---
## Section 7: Play the Game!

**Run this cell to test your solver against the AI**

In [8]:
STUDENT_TOKEN = 'gregfu'  # e.g., 'JOHN-DOE'
SOLVER = my_agent
MULTIPLAYER = False
MATCH_ID = None
NUM_GAMES = 1

result = play_game(
    solver=SOLVER,
    base_url=BASE_URL,
    token=STUDENT_TOKEN,
    game_type='quoridor',
    game_class=QuoridorGame,
    multiplayer=MULTIPLAYER,
    match_id=MATCH_ID,
    num_games=NUM_GAMES,
    debug=False,
    verbose=True
)

stats, all_results = result
print("\n📊 Summary:")
print(f"   Record: {stats['wins']}W - {stats['losses']}L - {stats['draws']}D")
print(f"   Win Rate: {stats['win_rate']*100:.1f}%")

🎮 Creating new match: 1 x quoridor
   Match ID: 1504
🔗 Joining match 1504...
   You are player: 1

🎮 GAME 1/1


Player 1: [0, 4]  Player 2: [8, 4]
Walls remaining - You: 10, Opponent: 10
·   ·   ·   ·   1   ·   ·   ·   ·
                                 
·   ·   ·   ·   ·   ·   ·   ·   ·
                                 
·   ·   ·   ·   ·   ·   ·   ·   ·
                                 
·   ·   ·   ·   ·   ·   ·   ·   ·
                                 
·   ·   ·   ·   ·   ·   ·   ·   ·
                                 
·   ·   ·   ·   ·   ·   ·   ·   ·
                                 
·   ·   ·   ·   ·   ·   ·   ·   ·
                                 
·   ·   ·   ·   ·   ·   ·   ·   ·
                                 
·   ·   ·   ·   2   ·   ·   ·   ·

🤔 Your turn (Player 2)...
   Moving pawn to (7, 4)

Player 1: [1, 4]  Player 2: [7, 4]
Walls remaining - You: 10, Opponent: 10
·   ·   ·   ·   ·   ·   ·   ·   ·
                                 
·   ·   ·   ·   1   ·   ·   ·   ·
     

In [10]:
'''
STUDENT_TOKEN = 'gregfu05'
MULTIPLAYER = True
MATCH_ID = '720'
NUM_GAMES = 1

# Use your agent this time
SOLVER = my_agent

result = play_game(
    solver=SOLVER,
    base_url=BASE_URL,
    token=STUDENT_TOKEN,
    game_type='quoridor',
    game_class=QuoridorGame,
    multiplayer=MULTIPLAYER,
    match_id=MATCH_ID,
    num_games=NUM_GAMES,
    debug=False,
    verbose=True
)
'''

"\nSTUDENT_TOKEN = 'gregfu05'\nMULTIPLAYER = True\nMATCH_ID = '720'\nNUM_GAMES = 1\n\n# Use your agent this time\nSOLVER = my_agent\n\nresult = play_game(\n    solver=SOLVER,\n    base_url=BASE_URL,\n    token=STUDENT_TOKEN,\n    game_type='quoridor',\n    game_class=QuoridorGame,\n    multiplayer=MULTIPLAYER,\n    match_id=MATCH_ID,\n    num_games=NUM_GAMES,\n    debug=False,\n    verbose=True\n)\n"

In [ ]:
STUDENT_TOKEN = "gregfu"
MULTIPLAYER = True
MATCH_ID = None       # Create new match
NUM_GAMES = 10

SOLVER = my_agent     # Your own agent plays as Player 1

result = play_game(
    solver=SOLVER,
    base_url=BASE_URL,
    token=STUDENT_TOKEN,
    game_type='quoridor',
    game_class=QuoridorGame,
    multiplayer=MULTIPLAYER,
    match_id=MATCH_ID,
    num_games=NUM_GAMES,
    debug=False,
    verbose=True
)
